MCP server, klient + mcp interceptor do obsługi errorów.<br>

toole <br>
state class<br>
podagenci wraz z dokładnymi system promptami.<br>

calle subagentów w poostaci tooli<br>
tool do updatu statu.<br>
główny agent, agregujący inne toole<br>

invokowanie agenta<br>

In [13]:
from dotenv import load_dotenv
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command
from tavily import TavilyClient
from typing import Dict, Any
from datetime import datetime


load_dotenv()

True

Flight agent, Venue agent - looks for places, playlist agent, chef agent

In [14]:
import asyncio

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent

RETRYABLE_MCP_CODES = {-32603}


class RetryMCPInterceptor:
    """Intercept MCP tool calls: retry transient failures, surface all errors gracefully.

    - Retryable McpError codes (e.g. -32603): retry with exponential backoff.
    - Non-retryable McpError codes (e.g. -32602): return error message immediately.
    - Any other exception (fetch failed, network errors, etc.): retry then return error message.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(code {exc.error.code}, attempt {attempt + 1}/{self.max_retries}): {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool call failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(attempt {attempt + 1}/{self.max_retries}): {exc}")

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[
                TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )


client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
        }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

travel_tools = await client.get_tools()

In [15]:
tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """
    Search web for information. Track your search count, by providing search_number (start at 1), and max_search_number at every call.
    Queries must only use plain text characters. don't use special characters. (e.g. 'e' instead of 'ę')
    """
    if search_number == max_search_number:
        return {"message": "Search limit reached. Summarise your data, and provide final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

In [16]:
from langchain.agents import AgentState
class WeddingState(AgentState):
    origins: list[str]
    destination: str
    guest_count: int
    date: datetime
    cost: int

In [17]:
model = 'gpt-5-nano'


flight_prompt = """
    You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Date for found wedding venue time.
    - Price (lowest, economy class)
    - Duration (shortest)
    To make things easy, only look for one ticket, one way.
    Look for tickets from all specified origins.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origins and destination. It is your job to think critically about the best options.
    If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again.
    Once you have found the best options, let the user know your shortlist of options, for each origin.
    """
venue_prompt = """
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options.
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """
dj_prompt = """
    You are wedding organiser focusing on finding deejay. Search internet, too look deejay around given location.
    While searching focus on following criteria:
    - Price (lowest)
    - Reviews (highest)
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """

flight_agent = create_agent(
    model=model,
    tools=travel_tools,
    system_prompt=flight_prompt
)

venue_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=venue_prompt
)

dj_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=dj_prompt
)

Call subagents

In [21]:

@tool
async def call_flight_subagent(runtime: ToolRuntime) -> str:
    """Call flight agent to find flights to given place, from specified cities in given date."""
    origins = runtime.state["origins"]
    destination = runtime.state["destination"]
    date = runtime.state["date"]
    response = await flight_agent.ainvoke({"messages": [HumanMessage(content=f"Find flights from {origins} to {destination}, on a date of {date}")]})
    return response["messages"][-1].content

@tool
def call_venue_subagent(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    guest_count = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {guest_count} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def call_dj_subagent(runtime: ToolRuntime) -> str:
    """Find deejay for wedding around specified place at specified date."""
    destination = runtime.state["destination"]
    query = f"Find deejay around {destination}, for a wedding"
    response = dj_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def update_state(origins: list[str], destination: str, guest_count: str, date: datetime, cost: int, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origins, destination, guest_count, date, cost.
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origins": origins,
        "destination": destination,
        "guest_count": guest_count,
        "date": date,
        "cost": cost,
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]
    })



In [39]:
from langgraph.checkpoint.memory import InMemorySaver
sys_prompt = """
    You are a wedding coordinator.
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks
    to your specialists for flights, venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.
    """

main_agent = create_agent(
    model=model,
    tools=[call_flight_subagent, call_venue_subagent, call_dj_subagent, update_state],
    state_schema=WeddingState,
    system_prompt=sys_prompt,
    checkpointer=InMemorySaver()
)

In [40]:
prompt = "Organise me wedding for 100 people in Gdansk in Poland. Guests will come from Warsaw and Poznan. Let the wedding start as soon as possible after 1st may of 2026."
response = await main_agent.ainvoke(
    {"messages": [HumanMessage(content=prompt)]},
    config={"tags": ["WP"], "recursion_limit": 40, "configurable": {"thread_id": "1"}}
)

[MCP interceptor] McpError on search-flight (code -32602, attempt 1/3): MCP error -32602: MCP error -32602: Invalid arguments for tool search-flight: [
  {
    "code": "invalid_type",
    "expected": "string",
    "received": "null",
    "path": [
      "returnDate"
    ],
    "message": "Expected string, received null"
  }
]
[MCP interceptor] McpError on search-flight (code -32602, attempt 1/3): MCP error -32602: MCP error -32602: Invalid arguments for tool search-flight: [
  {
    "code": "invalid_type",
    "expected": "string",
    "received": "null",
    "path": [
      "returnDate"
    ],
    "message": "Expected string, received null"
  }
]


In [41]:
print(response["messages"][-1].content)

Great news — I’ve set up your wedding profile and pulled together strong options for flights, venues, and a DJ in Gdańsk for 100 guests coming from Warsaw and Poznań, with the event slated to start as soon as possible after May 1, 2026 (date currently set to 2 May 2026 at 12:00 local time).

What I have now (summary)
- Dates/people: 100 guests from Warsaw and Poznań to Gdańsk. Wedding start time currently noted as 12:00 on 2 May 2026.
- Flights: Direct options from Warsaw to Gdańsk (WAW → GDN) with ~55 minutes flight time, around 94 EUR.
  - 10:25–11:20
  - 13:15–14:10
  - 19:55–20:50
- Poznań options: Several with longer travel times or one-stop routes; some aren’t ideal for a tight wedding day schedule. We can consider a group transfer by road/train from Poznań to Gdańsk if air travel isn’t practical.
- Venues (two strong fits for 100 guests in Gdańsk):
  - Sala Weselna Ołowianka (Gdańsk) — riverside historic setting, capacity up to 120, typical menu pricing from ~200 PLN per person 

In [42]:
response = await main_agent.ainvoke(
    {"messages": [HumanMessage(content="Sala weselna olowianka is good. Ceremony can be done around 12-13. For deejay pick budged-friendly. Dont book any flights, just leave info about best flights, once you set a date. Leave catering for now")]},
    config={"tags": ["WP"], "recursion_limit": 40, "configurable": {"thread_id": "1"}}
)

In [43]:
print(response["messages"][-1].content)


Perfect. Here’s the plan based on your updates:

Confirmed details
- Venue: Sala Weselna Ołowianka in Gdańsk. Capacity fits 100 guests.
- Ceremony window: around 12:00–13:00.
- Guest origin: 100 guests (Warsaw and Poznań).
- Catering: on hold for now.
- Flights: not booked yet; you’ll get flight options later when you want to finalize date and arrivals.

What I’ll do next (and what I need from you)
1) Venue inquiry for Sala Weselna Ołowianka
- I’ll reach out to check:
  - Availability for May 2, 2026, with ceremony around 12:00–13:00 and reception following.
  - What the venue package includes (tables, chairs, basic décor, AV, dance floor, cake table, etc.).
  - Capacity confirmation for 100 guests in your chosen setup.
  - Deposit schedule and cancellation terms.
- I’ll avoid discussing catering specifics until you’re ready to proceed with menus.

2) Budget-friendly DJ options (shortlist, with outreach)
- I’ll contact 2–3 budget-friendly options and request quotes for:
  - 6–8 hours o